<a href="https://colab.research.google.com/github/andresecha/Methodologie-de-la-recherche-Numerique/blob/main/OCR/Mistral/Metodological_guide_MistralOCR_Transcripcion_ARIANE_FR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📄 Comment convertir des documents imprimés et certains types de manuscrits en textes exploitables ?

## Guide méthodologique pour la reconnaissance optique de documents avec Mistral OCR *(Mistral AI)*

Andrés Echavarría et Théo Roulet

**Groupe d'intérêt « Acquisition de textes assistée par ordinateur »** · Consortium Huma-Num **ARIANE**

---

### Problématique

Les sciences humaines et sociales utilisent de nombreux documents contenant du texte, mais non directement exploitables comme tel : articles numérisés, archives photographiées, manuscrits, actes administratifs, plans annotés, coupures de presse, partitions, formulaires historiques ou encore PDF «image», dont le texte n’est ni sélectionnable ni interrogeable.

Sous cette forme, ces documents restent difficilement accessibles aux outils d’analyse textuelle, d’indexation et de recherche, pourtant centraux en humanités numériques. La reconnaissance optique de caractères (OCR) apporte une solution partielle, mais elle demeure limitée : elle extrait le texte au prix d’une perte fréquente de la structure du document (titres, tableaux, équations, illustrations) et s’avère peu fiable pour des documents complexes ou multilingues.

Se pose dès lors une question opérationnelle centrale :

> *Dans quelle mesure les outils actuels d’OCR fondés sur l’intelligence artificielle permettent-ils d’extraire de manière fiable non seulement le texte, mais aussi la structure des documents (titres, tableaux, équations, images), à partir de sources complexes (imprimées, manuscrites ou numérisées) et multilingues ? Par ailleurs, comment ces résultats peuvent-ils être intégrés dans des chaînes de traitement adaptées à la recherche ?*
---
### Ce que propose ce cahier
Ce cahier Google Colab propose un **guide pratique et structuré** pour l’utilisation de [**Mistral OCR**](https://mistral.ai/news/mistral-ocr), l’API de reconnaissance optique de documents développée par [Mistral AI](https://mistral.ai) (entreprise française). Contrairement aux OCR traditionnels, Mistral OCR ne se limite pas à l’extraction du texte : il s’agit d’un **modèle de compréhension documentaire** capable de prendre en compte la structure, la mise en page et les éléments multimodaux des documents.


| Capacité | Description | Intérêt pour la recherche |
|----------|-------------|---------------------------|
| **Extraction de texte structuré** | Génère du texte en Markdown en conservant titres, listes, paragraphes et liens | Produire des versions textuelles respectant l’organisation du document |
| **Reconnaissance de tableaux** | Extrait les tableaux au format Markdown ou HTML | Exploiter des données tabulaires issues de rapports, recensements ou catalogues |
| **Extraction d'équations** | Identifie les expressions mathématiques et les restitue en LaTeX | Numériser des articles scientifiques et techniques en préservant les formules |
| **Extraction d'images insérées** | Identifie et extrait les images contenues dans le document | Préserver les formules dans les documents scientifiques et techniques |
| **Compréhension multilingue** | Prend en charge de nombreuses langues et systèmes d’écriture | TTraiter des corpus multilingues (arabe, chinois, grec, hébreu, devanagari, etc.) |
| **Sortie structurée (JSON)** | Permet de définir des schémas (Pydantic) pour extraire des champs spécifiques | Automatiser l’extraction de métadonnées (auteur, titre, date, résumé) |
| **Annotations** | Génère des descriptions et classifications du document et de ses éléments | Enrichir automatiquement les métadonnées d’un corpus |
---
### Architecture du guide

Ce guide est organisé en sections couvrant l’ensemble de la chaîne de traitement d’un processus d’OCR : :

1. **Configuration de l'environnement** — Installation du SDK Mistral et configuration de la clé d’API.
2. **OCR de base d'un PDF** — Traitement d’un PDF (URL, Google Drive ou téléversement) et extraction en Markdown.
3. **OCR d'images** — Traiter d'images (`.png`, `.jpg`, etc.).
4. **Visualisation des résultats** — Affichage du Markdown avec les images extraites.
5. **Extraction de tableaux** — Récupération en Markdown ou HTML.
6. **Sortie structurée avec Pydantic** — Définition de schémas et extraction de champs (langue, résumé, auteurs, etc.)
7. **Annotations de document et de régions** — Utilisation de `document_annotation` et `bbox_annotation` pour enrichir les métadonnées.
8. **Traitement par lots** — Automatisation sur plusieurs documents avec sauvegarde (Google Drive).
9. **Conseils et résolution de problèmes** — Limites de l'API (50 Mo, 1 000 pages), erreurs, qualité des sources et recommandations pratiques.
---
### Conditions d'utilisation

- **Environnement recommandé** : Google Colab (aucun GPU requis ; traitement effectué dans le cloud Mistral AI).
- **Prérequis** : Disposer d'une **clé d'API Mistral** (gratuite pour les tests sur [console.mistral.ai](https://console.mistral.ai)) (*environ 1000 pages par jour en usage gratuit*).
- AUTREMENT, le **coût de l'API** : Environ **1 000 pages par dollar** (tarif standard), ou 2 000 pages par dollar en mode batch.
- **Formats d'entrée** : PDF, PPTX, DOCX et formats image (PNG, JPEG, AVIF, etc.).
- **Modèle utilisé** : `mistral-ocr-latest` (actuellement Mistral OCR 3, décembre 2025).
- **Limites actuelles** : 50 Mo par fichier et 1 000 pages par requête.


⚠️ **Avant de commencer :** vous devez disposer d’une clé API Mistral. Si ce n’est pas le cas, rendez-vous sur [console.mistral.ai](https://console.mistral.ai), créez un compte et générez une clé dans la section **API Keys**. 🚩🚩**Ne partagez jamais cette clé avec personne.**🚩🚩


---
---
### Comment citer ce guide

> Echavarría, Andrés. [demander affiliation à Paco, Mathilde, Bárbara, y David]. *Guide méthodologique pour la reconnaissance optique de documents avec Mistral OCR (Mistral AI)*.Groupe d'intérêt « Acquisition automatique de textes », Consortium Huma-Num ARIANE. [Carnet Jupyter Google Colab], version 1.0, mars 2026.
---
---


> 💡 **Note** : Ce guide est un document évolutif. Les contributions, corrections et suggestions d’usages sont les bienvenues via le groupe d’intérêt.


## ⚙️ ÉTAPE 1 — Installation de la bibliothèque

Exécutez cette cellule **une seule fois**. Elle installe le paquet Python `mistralai` qui permet de communiquer avec l'API Mistral.

> 💡 Si vous travaillez sur **Google Colab**, la cellule s'exécute directement. Si vous travaillez en local, assurez-vous d'avoir Python 3.8+ et pip installés.

In [ ]:
# Installation du paquet mistralai
# Le point d'exclamation ! indique à Jupyter d'exécuter une commande système (comme dans un terminal)
!pip install mistralai

## 📦 ÉTAPE 2 — Importation des outils

On charge ici toutes les bibliothèques dont on aura besoin. Exécutez cette cellule sans rien modifier.

In [ ]:
import os
import json
import base64
from pathlib import Path
from mistralai.client import Mistral # Regarder la documentation si ici ne marche pas est probablement le nom de la librairie
from IPython.display import display, Markdown, Image

print('✅ Bibliothèques chargées avec succès !')

## 🔑 ÉTAPE 3 — Votre clé API Mistral

C'est ici que vous saisissez votre clé API. Elle ressemble à une longue suite de caractères du type : `sk-abc123XYZ...`

**Deux options sont proposées ci-dessous :**

- **Option A (recommandée sur Google Colab)** : utilise le coffre-fort sécurisé de Colab (`userdata`). Allez dans le menu ▶ *Runtime* → *Secrets* (🔑) et ajoutez une entrée avec le nom `MISTRAL_API_KEY` et votre clé comme valeur.

- **Option B (en local ou si Option A échoue)** : saisissez directement votre clé dans le code. ⚠️ Ne partagez jamais un notebook contenant une clé en clair !

In [ ]:
# ─────────────────────────────────────────────────────────────
# OPTION A : Clé stockée dans les secrets Colab (recommandé)
# ─────────────────────────────────────────────────────────────
try:
    from google.colab import userdata
    MISTRAL_API_KEY = userdata.get('MISTRAL_API_KEY')
    print('✅ Clé API chargée depuis les secrets Colab.')

# ─────────────────────────────────────────────────────────────
# OPTION B : Saisie manuelle (remplacez le texte entre guillemets)
# ─────────────────────────────────────────────────────────────
except Exception:
    MISTRAL_API_KEY = 'COLLEZ_VOTRE_CLÉ_ICI'  # ← remplacez cette valeur
    print('⚠️  Clé API saisie manuellement.')

# Création du client Mistral — ne modifiez pas cette ligne
client = Mistral(api_key=MISTRAL_API_KEY)
print('✅ Client Mistral prêt à l\'emploi !')

---

# 🌐 MODE A — OCR d'un PDF disponible en ligne (via une URL)

Utilisez ce mode si votre document est accessible par un lien web (ex. : un article en open access, un rapport public, un PDF hébergé sur un serveur).

> Si votre fichier est sur votre ordinateur, passez directement au **MODE B** plus bas.

## 🔗 ÉTAPE A1 — Renseigner l'URL du document

Remplacez l'URL d'exemple par celle de votre propre document PDF. L'URL doit commencer par `https://` et pointer directement vers un fichier `.pdf`.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ✏️  ZONE À MODIFIER : remplacez l'URL ci-dessous par celle de votre document
# ─────────────────────────────────────────────────────────────────────────────

PDF_URL = 'https://www.bcub.ro/lib2life/Brutu_Heliade-Radulescu%20Ion_Bucuresci_1878.pdf'  # ← URL de votre PDF

# Nom du fichier de sortie (sans extension) — modifiez si vous le souhaitez
NOM_SORTIE = 'resultat_ocr_url'

print(f'📄 Document cible : {PDF_URL}')

## 🚀 ÉTAPE A2 — Lancement de l'OCR

Cette cellule envoie le document à Mistral OCR et récupère le résultat. Le traitement peut prendre quelques secondes à une minute selon la taille du PDF.

**Ce que fait le code :**
- `model='mistral-ocr-latest'` → utilise le dernier modèle OCR de Mistral
- `type='document_url'` → indique qu'on envoie un lien (et non un fichier)
- `include_image_base64=True` → demande à récupérer aussi les images du document

In [ ]:
# Envoi du document à l'API Mistral OCR
# Ne modifiez pas ce bloc — il fonctionne tel quel

print('⏳ OCR en cours, patientez...')

ocr_response = client.ocr.process(
    model='mistral-ocr-latest',
    document={
        'type': 'document_url',
        'document_url': PDF_URL
    },
    include_image_base64=True   # mettre False si vous ne voulez pas les images
)

nb_pages = len(ocr_response.pages)
print(f'✅ OCR terminé ! Nombre de pages traitées : {nb_pages}')

## 👁️ ÉTAPE A3 — Prévisualisation du résultat

On affiche ici les **3 premières pages** pour vérifier que l'OCR a bien fonctionné. Le texte est affiché en Markdown (titres, listes, tableaux sont préservés).

> Modifiez `NB_PAGES_APERCU` si vous voulez voir plus ou moins de pages.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ✏️  ZONE À MODIFIER : nombre de pages à afficher en aperçu (défaut : 3)
# ─────────────────────────────────────────────────────────────────────────────
NB_PAGES_APERCU = 5

# Affichage des pages en Markdown rendu
for i, page in enumerate(ocr_response.pages[:NB_PAGES_APERCU]):
    print(f'\n--- PAGE {i + 1} ---')
    display(Markdown(page.markdown))

## 💾 ÉTAPE A4 — Sauvegarde des résultats

On sauvegarde le texte extrait dans **3 formats** :

- **`.txt`** → texte brut, toutes pages concaténées, facile à lire dans n'importe quel éditeur
- **`.md`** → fichier Markdown, avec la mise en forme préservée (titres, tableaux…)
- **`.json`** → format structuré, une entrée par page, utile pour un traitement ultérieur

Les fichiers sont sauvegardés dans le répertoire courant du notebook.

In [ ]:
# ── Sauvegarde en .txt (texte brut) ──
texte_complet = '\n\n'.join(page.markdown for page in ocr_response.pages)
with open(f'{NOM_SORTIE}.txt', 'w', encoding='utf-8') as f:
    f.write(texte_complet)
print(f'✅ Texte sauvegardé : {NOM_SORTIE}.txt')

# ── Sauvegarde en .md (Markdown) ──
with open(f'{NOM_SORTIE}.md', 'w', encoding='utf-8') as f:
    for i, page in enumerate(ocr_response.pages):
        f.write(f'<!-- PAGE {i+1} -->\n')
        f.write(page.markdown + '\n\n')
print(f'✅ Markdown sauvegardé : {NOM_SORTIE}.md')

# ── Sauvegarde en .json (structuré, une entrée par page) ──
data_json = [
    {
        'page': i + 1,
        'texte': page.markdown,
        'nb_images': len(page.images) if page.images else 0
    }
    for i, page in enumerate(ocr_response.pages)
]
with open(f'{NOM_SORTIE}.json', 'w', encoding='utf-8') as f:
    json.dump(data_json, f, ensure_ascii=False, indent=2)
print(f'✅ JSON sauvegardé : {NOM_SORTIE}.json')

## 🖼️ ÉTAPE A5 (optionnel) — Extraction des images du document

Si le PDF contient des images (illustrations, figures, tableaux scannés…), cette cellule les sauvegarde dans un dossier `images_<NOM_SORTIE>/`.

> **Conseil :** si votre document ne contient pas d'images ou si vous n'en avez pas besoin, vous pouvez passer cette cellule.

In [ ]:
# Création du dossier de destination pour les images
dossier_images = Path(f'images_{NOM_SORTIE}')
dossier_images.mkdir(exist_ok=True)

compteur_images = 0

for i, page in enumerate(ocr_response.pages):
    if not page.images:
        continue  # pas d'images sur cette page, on passe
    for img in page.images:
        if not img.image_base64:
            continue  # image sans données base64, on ignore

        # Décodage de l'image depuis le format base64
        donnees_image = base64.b64decode(img.image_base64)

        # Nom du fichier image : page_XX_imgYY.png
        nom_image = f'page_{i+1:02d}_img{compteur_images+1:02d}.png'
        chemin_image = dossier_images / nom_image

        with open(chemin_image, 'wb') as f:
            f.write(donnees_image)

        compteur_images += 1
        print(f'  🖼️  Image sauvegardée : {chemin_image}')

if compteur_images == 0:
    print('ℹ️  Aucune image trouvée dans ce document.')
else:
    print(f'\n✅ {compteur_images} image(s) sauvegardée(s) dans le dossier : {dossier_images}/')

---

# 📁 MODE B — OCR d'un fichier PDF local (depuis votre ordinateur)

Utilisez ce mode si le document se trouve sur votre ordinateur (ou dans votre Google Drive). Le processus comporte une étape supplémentaire : **on téléverse d'abord le fichier vers les serveurs Mistral**, puis on lance l'OCR.

> ⚠️ Le fichier envoyé à Mistral sera temporairement stocké sur leurs serveurs. Ne transmettez pas de documents confidentiels sans vérifier les conditions d'usage de Mistral AI.

## 📤 ÉTAPE B1 — Indiquer le chemin de votre fichier PDF

Deux sous-options :

**Sur Google Colab :** la cellule ouvre une boîte de dialogue pour charger le fichier directement depuis votre ordinateur. Cliquez sur le bouton qui apparaît après l'exécution.

**En local (Jupyter classique) :** renseignez manuellement le chemin complet vers votre fichier dans la variable `CHEMIN_PDF_LOCAL`.

> Exemple de chemin local : `'/Users/monnom/Documents/article.pdf'` (Mac/Linux) ou `'C:/Users/monnom/Documents/article.pdf'` (Windows)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ✏️  ZONE À MODIFIER selon votre environnement
# ─────────────────────────────────────────────────────────────────────────────

# Option B1a — Sur Google Colab : upload interactif
try:
    from google.colab import files
    print('📂 Colab détecté. Cliquez sur le bouton ci-dessous pour charger votre PDF...')
    fichiers_uploades = files.upload()
    # Récupère automatiquement le nom du fichier uploadé
    CHEMIN_PDF_LOCAL = list(fichiers_uploades.keys())[0]
    print(f'✅ Fichier reçu : {CHEMIN_PDF_LOCAL}')

# Option B1b — En local : indiquer le chemin manuellement
except Exception:
    CHEMIN_PDF_LOCAL = '/home/roulett/dev/test_mistral_ocr/aix/aix_4p_mistral.pdf'  # ← remplacez par votre chemin
    print(f'📄 Fichier local sélectionné : {CHEMIN_PDF_LOCAL}')

# Nom de sortie basé sur le nom du fichier (sans extension)
NOM_SORTIE_LOCAL = Path(CHEMIN_PDF_LOCAL).stem
print(f'📝 Les résultats seront sauvegardés sous le nom : {NOM_SORTIE_LOCAL}.*')

## ☁️ ÉTAPE B2 — Envoi du fichier à Mistral

Avant de lancer l'OCR, on **téléverse le fichier** vers l'API Mistral. L'API retourne un identifiant unique (`file_id`) qui sera utilisé à l'étape suivante.

Ce processus peut prendre quelques secondes selon la taille du fichier et votre connexion.

In [ ]:
# Ouverture et envoi du fichier PDF à Mistral
print(f'⏳ Envoi du fichier vers Mistral : {CHEMIN_PDF_LOCAL}...')

with open(CHEMIN_PDF_LOCAL, 'rb') as f:
    fichier_uploade = client.files.upload(
        file={
            'file_name': Path(CHEMIN_PDF_LOCAL).name,  # nom du fichier
            'content': f                               # contenu binaire
        },
        purpose='ocr'  # indique à Mistral que ce fichier est destiné à l'OCR
    )

FILE_ID = fichier_uploade.id
print(f'✅ Fichier envoyé avec succès !')
print(f'🆔 Identifiant Mistral du fichier : {FILE_ID}')

## 🚀 ÉTAPE B3 — Lancement de l'OCR sur le fichier local

Même logique qu'en Mode A, mais on utilise cette fois `type='document_url'` avec une URL signée générée automatiquement à partir du `FILE_ID`.

**Options utiles que vous pouvez activer :**

- `pages=[0, 1, 2]` → traiter uniquement les pages 1, 2 et 3 (indices démarrant à 0)
- `include_image_base64=False` → ne pas récupérer les images (plus rapide)
- `image_limit=5` → limiter à 5 images par page maximum

In [ ]:
# Récupération de l'URL signée pour accéder au fichier uploadé
url_signee = client.files.get_signed_url(file_id=FILE_ID)

# ─────────────────────────────────────────────────────────────────────────────
# ✏️  OPTIONS À MODIFIER selon vos besoins (décommentez ce que vous souhaitez)
# ─────────────────────────────────────────────────────────────────────────────

# pages=[0, 1, 2]    → traite uniquement les pages 1-3 (décommenter si besoin)
# image_limit=10     → max 10 images par page (décommenter si besoin)

print('⏳ OCR en cours, patientez...')

ocr_response_local = client.ocr.process(
    model='mistral-ocr-latest',
    document={
        'type': 'document_url',
        'document_url': url_signee.url
    },
    include_image_base64=True   # mettre False si vous ne voulez pas les images
    # pages=[0, 1, 2],          # décommentez pour traiter seulement certaines pages
    # image_limit=10,            # décommentez pour limiter le nb d'images
)

nb_pages_local = len(ocr_response_local.pages)
print(f'✅ OCR terminé ! Nombre de pages traitées : {nb_pages_local}')

## 👁️ ÉTAPE B4 — Prévisualisation du résultat

Vérification rapide : on affiche les premières pages pour s'assurer que l'OCR a bien tourné.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ✏️  ZONE À MODIFIER : nombre de pages à afficher en aperçu
# ─────────────────────────────────────────────────────────────────────────────
NB_PAGES_APERCU_LOCAL = 3

for i, page in enumerate(ocr_response_local.pages[:NB_PAGES_APERCU_LOCAL]):
    print(f'\n--- PAGE {i + 1} ---')
    display(Markdown(page.markdown))

## 💾 ÉTAPE B5 — Sauvegarde des résultats

Identique au Mode A : on sauvegarde en `.txt`, `.md` et `.json`.

> Les fichiers portent le nom de votre PDF original (sans l'extension `.pdf`).

In [ ]:
# ── Sauvegarde en .txt (texte brut) ──
texte_complet_local = '\n\n'.join(page.markdown for page in ocr_response_local.pages)
with open(f'{NOM_SORTIE_LOCAL}.txt', 'w', encoding='utf-8') as f:
    f.write(texte_complet_local)
print(f'✅ Texte sauvegardé : {NOM_SORTIE_LOCAL}.txt')

# ── Sauvegarde en .md (Markdown) ──
with open(f'{NOM_SORTIE_LOCAL}.md', 'w', encoding='utf-8') as f:
    for i, page in enumerate(ocr_response_local.pages):
        f.write(f'<!-- PAGE {i+1} -->\n')
        f.write(page.markdown + '\n\n')
print(f'✅ Markdown sauvegardé : {NOM_SORTIE_LOCAL}.md')

# ── Sauvegarde en .json (structuré, une entrée par page) ──
data_json_local = [
    {
        'page': i + 1,
        'texte': page.markdown,
        'nb_images': len(page.images) if page.images else 0
    }
    for i, page in enumerate(ocr_response_local.pages)
]
with open(f'{NOM_SORTIE_LOCAL}.json', 'w', encoding='utf-8') as f:
    json.dump(data_json_local, f, ensure_ascii=False, indent=2)
print(f'✅ JSON sauvegardé : {NOM_SORTIE_LOCAL}.json')

## 🖼️ ÉTAPE B6 (optionnel) — Extraction des images du document local

Même principe qu'en Mode A : on récupère les images et on les sauvegarde dans un dossier dédié.

In [ ]:
# Création du dossier de destination pour les images
dossier_images_local = Path(f'images_{NOM_SORTIE_LOCAL}')
dossier_images_local.mkdir(exist_ok=True)

compteur_images_local = 0

for i, page in enumerate(ocr_response_local.pages):
    if not page.images:
        continue
    for img in page.images:
        if not img.image_base64:
            continue

        donnees_image = base64.b64decode(img.image_base64)
        nom_image = f'page_{i+1:02d}_img{compteur_images_local+1:02d}.png'
        chemin_image = dossier_images_local / nom_image

        with open(chemin_image, 'wb') as f:
            f.write(donnees_image)

        compteur_images_local += 1
        print(f'  🖼️  Image sauvegardée : {chemin_image}')

if compteur_images_local == 0:
    print('ℹ️  Aucune image trouvée dans ce document.')
else:
    print(f'\n✅ {compteur_images_local} image(s) sauvegardée(s) dans : {dossier_images_local}/')

## 🗑️ ÉTAPE B7 (recommandé) — Suppression du fichier sur les serveurs Mistral

Par mesure d'hygiène et pour éviter des coûts inutiles, il est conseillé de supprimer le fichier de vos espaces de stockage Mistral une fois l'OCR terminé.

In [ ]:
# Suppression du fichier uploadé sur les serveurs Mistral
client.files.delete(file_id=FILE_ID)
print(f'✅ Fichier supprimé des serveurs Mistral (ID : {FILE_ID})')

---

## ✅ Récapitulatif des fichiers générés

Exécutez cette cellule pour voir la liste de tous les fichiers créés par ce notebook.

In [ ]:
import glob

print('📂 Fichiers générés dans ce répertoire :\n')

# Fichiers texte, markdown, JSON
for ext in ['*.txt', '*.md', '*.json']:
    for f in sorted(glob.glob(ext)):
        taille = Path(f).stat().st_size
        print(f'  📄 {f}  ({taille:,} octets)')

# Dossiers d'images
for dossier in sorted(glob.glob('images_*')):
    nb = len(list(Path(dossier).glob('*.png')))
    print(f'  📁 {dossier}/  ({nb} image(s))')

---

## 💬 Besoin d'aide ? Questions fréquentes

| Problème | Solution |
|----------|---------|
| `AuthenticationError` | Votre clé API est invalide ou mal saisie → vérifiez l'Étape 3 |
| `FileNotFoundError` | Le chemin vers votre PDF est incorrect → vérifiez l'Étape B1 |
| Le texte est vide | Le PDF est peut-être protégé ou illisible → vérifiez le fichier source |
| Pas d'images extraites | Vérifiez que `include_image_base64=True` est bien activé |
| L'OCR est lent | Les gros PDF prennent plus de temps ; essayez de limiter avec `pages=[0,1,2]` |

---

*Carnet de travail rédigé par le groupe d'intérêt sur l'acquisition assistée de textes du **consortium ARIANE**— Mars 2026*  
*Outil basé sur [Mistral OCR API](https://mistral.ai/fr/news/mistral-ocr)*